# Baseline CNN — 16 ch/hand · 2000 Hz · Log-Spectrogram

**Model:** TDSConvCTC (5.3 M params)  
**Pipeline:** `ToTensor → BandRotation → TemporalJitter → LogSpectrogram → SpecAugment`  
**Results:** Val CER **18.52%** · Test CER **22.28%** · Training ~3 h 51 m

```
source /home/benforbes/emg2qwerty/venv/bin/activate
cd ~/C247_mumbikaijonathanben && jupyter notebook
```

## 1. Setup

In [ ]:
import os, sys, subprocess, glob
from pathlib import Path

REPO_ROOT = Path(os.getcwd())
while not (REPO_ROOT / 'emg2qwerty').is_dir():
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f'Repo root: {REPO_ROOT}')

## 2. Patch `emg2qwerty` with Playground_Ben Modifications

Needed for the dynamic channel count in `lightning.py` and the new transforms in `transforms.py`.

In [ ]:
import shutil

PLAYGROUND = REPO_ROOT / 'Playground_Ben'
shutil.copy(PLAYGROUND / 'emg2qwerty/transforms.py', REPO_ROOT / 'emg2qwerty/transforms.py')
shutil.copy(PLAYGROUND / 'emg2qwerty/lightning.py',  REPO_ROOT / 'emg2qwerty/lightning.py')
print('Patched: transforms.py, lightning.py')

## 3. Train

In [ ]:
result = subprocess.run(
    [sys.executable, '-m', 'emg2qwerty.train', 'user=single_user'],
    cwd=str(REPO_ROOT)
)
print(f'Exit code: {result.returncode}')

## 4. Load Results & Plot

In [ ]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator, SCALARS
import numpy as np, matplotlib.pyplot as plt

UCLA = {'blue': '#2774AE', 'gold': '#FFD100', 'dark_blue': '#003B5C', 'dark_gold': '#FFB300'}

# Auto-detect most recent log dir
logs = sorted((REPO_ROOT / 'logs').glob('*/*'), reverse=True)
log_dir = next(d for d in logs if (d / 'lightning_logs' / 'version_0').exists())
print(f'Log dir: {log_dir}')

# Load TensorBoard scalars
scalars: dict = {}
for ef in sorted((log_dir / 'lightning_logs' / 'version_0').glob('events.out.tfevents.*')):
    ea = EventAccumulator(str(ef), size_guidance={SCALARS: 0})
    ea.Reload()
    for tag in ea.Tags().get('scalars', []):
        scalars.setdefault(tag, []).extend(ea.Scalars(tag))

def epoch_series(scalars, tag):
    s2e = {e.step: int(e.value) for e in scalars.get('epoch', [])}
    pairs = sorted((s2e[e.step], e.value) for e in scalars.get(tag, []) if e.step in s2e)
    return (np.array([p[0] for p in pairs]), np.array([p[1] for p in pairs])) if pairs else (np.array([]), np.array([]))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('CNN Baseline — 16 ch/hand, 2000 Hz', fontsize=13)

ve, vc = epoch_series(scalars, 'val/CER')
axes[0].plot(ve, vc, color=UCLA['blue'], lw=2, label='Val CER')
axes[0].set(xlabel='Epoch', ylabel='CER (%)', title='Validation CER')
axes[0].set_ylim(bottom=0); axes[0].legend(); axes[0].grid(True, alpha=0.25, ls='--')

te, tl = epoch_series(scalars, 'train/loss')
le, vl = epoch_series(scalars, 'val/loss')
axes[1].plot(te, tl, color=UCLA['dark_blue'], lw=1.5, alpha=0.8, label='Train loss')
axes[1].plot(le, vl, color=UCLA['gold'],      lw=2,   label='Val loss')
axes[1].set(xlabel='Epoch', ylabel='CTC Loss', title='Train vs Val Loss')
axes[1].set_ylim(bottom=0); axes[1].legend(); axes[1].grid(True, alpha=0.25, ls='--')

plt.tight_layout(); plt.show()

## 5. Summary

In [ ]:
def last_val(scalars, tag):
    evts = scalars.get(tag, [])
    return evts[-1].value if evts else float('nan')

wall = [e.wall_time for evts in scalars.values() for e in evts]
hrs  = (max(wall) - min(wall)) / 3600 if wall else 0

print('=' * 38)
print('  CNN Baseline — 16 ch, 2000 Hz')
print('=' * 38)
print(f"  Val  CER  : {last_val(scalars, 'val/CER'):.2f}%")
print(f"  Test CER  : {last_val(scalars, 'test/CER'):.2f}%")
print(f"  Val  Loss : {last_val(scalars, 'val/loss'):.4f}")
print(f"  Training  : {hrs:.1f} hrs")
print('=' * 38)